Make sure the right schema is used

In [0]:
USE CATALOG sac;
USE SCHEMA customer_service;

# Gold Tables
average customer

In [0]:
CREATE OR REPLACE VIEW average_customer AS
SELECT
    zone,
    COUNT(*) AS amount_customers,
    ROUND(AVG(account_age_months), 2) AS avg_account_age,
    ROUND(AVG(monthly_bill), 2) AS avg_monthly_bill,
    ROUND(AVG(speed_tier_mbps), 0) AS avg_speed_tier,
    ROUND(AVG(data_usage_gb_last_month), 2) AS avg_data_usage
FROM
    customer
GROUP BY
    zone;

tickets per customer

In [0]:
CREATE OR REPLACE VIEW customer_connection_ticket_count AS
SELECT
    c.customer_id,
    COUNT(DISTINCT l.timestamp) AS connection_fails,
    COUNT(DISTINCT s.ticket_id) AS tickets,
    COUNT(DISTINCT ca.session_id) AS chats,
    ch.churned as churned
FROM
    customer c
        JOIN ticket s
            ON c.customer_id = s.customer_id
        LEFT JOIN log l
            ON c.customer_id = l.customer_id
            AND l.issue_detected NOT LIKE 'none'
        LEFT JOIN churn ch
            ON c.customer_id = ch.customer_id
        LEFT JOIN chat ca
            ON c.customer_id = ca.customer_id
GROUP BY
    c.customer_id, ch.churned;

churned customer

In [0]:
CREATE OR REPLACE VIEW churned_customer_details AS
SELECT
    c.customer_id,
    c.account_age_months,
    c.speed_tier_mbps,
    c.monthly_bill,
    COUNT(DISTINCT l.timestamp) AS connection_fails,
    COUNT(DISTINCT s.ticket_id) AS tickets,
    COUNT(DISTINCT ca.session_id) AS chats
FROM
    customer c
        LEFT JOIN ticket s
            ON c.customer_id = s.customer_id
        LEFT JOIN log l
            ON c.customer_id = l.customer_id
            AND l.issue_detected NOT LIKE 'none'
        LEFT JOIN chat ca
            ON c.customer_id = ca.customer_id
        JOIN churn ch
            ON c.customer_id = ch.customer_id
WHERE
    ch.churned = true
GROUP BY
    c.customer_id,
    c.account_age_months,
    c.speed_tier_mbps,
    c.monthly_bill;

location detail

In [0]:
CREATE OR REPLACE VIEW location_detail AS
WITH revenue_per_location AS (
    SELECT
        zone,
        SUM(monthly_bill) AS revenue
    FROM
        customer
    GROUP BY
        zone
),
issues_per_location AS (
    SELECT
        c.zone,
        COUNT(
            CASE
                WHEN l.issue_detected NOT LIKE 'none' THEN 1
            END
        ) AS issue_count
    FROM
        customer c
            LEFT JOIN log l
                ON c.customer_id = l.customer_id
    GROUP BY
        c.zone
)
SELECT
    c.zone,
    COUNT(DISTINCT c.customer_id) AS customer_count,
    ROUND(r.revenue / 1000, 2) AS revenue_in_t,
    i.issue_count AS issue_count,
    COUNT(DISTINCT t.ticket_id) AS ticket_count,
    COUNT(DISTINCT ch.session_id) AS chat_count
FROM
    customer c
        LEFT JOIN revenue_per_location r
            ON c.zone = r.zone
        LEFT JOIN issues_per_location i
            ON c.zone = i.zone
        LEFT JOIN ticket t
            ON c.customer_id = t.customer_id
        LEFT JOIN chat ch
            ON c.customer_id = ch.customer_id
GROUP BY
    c.zone,
    r.revenue,
    i.issue_count;

average connection quality

In [0]:
CREATE OR REPLACE VIEW average_connection_quality AS
SELECT
    c.zone,
    ROUND(AVG(l.speed_measured_mbps), 0) AS avg_speed,
    ROUND(AVG(l.packet_loss_percent), 2) AS avg_packet_loss,
    ROUND(AVG(l.latency_ms), 2) AS avg_latency,
    ROUND(AVG(l.downtime_minutes), 2) AS avg_downtime,
    ROUND(AVG(l.connection_drops_count), 2) AS avg_connection_drops,
    COUNT(
        CASE
            WHEN l.issue_detected NOT LIKE 'none' THEN 1
            ELSE 0
        END
    ) AS count_issues
FROM
    log l
        LEFT JOIN customer c
            ON l.customer_id = c.customer_id
GROUP BY
    zone;

chat issues

In [0]:
CREATE OR REPLACE VIEW chat_issues AS
SELECT
    c.classification,
    m.sentiment,
    COUNT(m.sentiment) AS count,
    FIRST(c.comment) AS exmp_comment
FROM
    chat c
    LEFT JOIN message m
WHERE
    classification IS NOT NULL
GROUP BY
    c.classification,
    m.sentiment
ORDER BY
    count DESC

sentiment for agent

In [0]:
CREATE OR REPLACE VIEW sentiment_for_agent AS
SELECT
    CONCAT(a.first_name, ' ', a.last_name) AS agent_name,
    m.sentiment,
    COUNT(m.sentiment) AS count
FROM
    chat c
        JOIN message m
            ON c.session_id = m.session_id
            AND m.speaker LIKE 'customer'
        LEFT JOIN agent a
            ON c.agent_id = a.agent_id
GROUP BY
    agent_name,
    m.sentiment;

# Show tables

In [0]:
SELECT * FROM average_customer;

In [0]:
SELECT * FROM customer_connection_ticket_count ORDER BY tickets DESC LIMIT 20;

In [0]:
SELECT * FROM churned_customer_details ORDER BY tickets DESC LIMIT 20;

In [0]:
SELECT * FROM location_detail ORDER BY customer_count DESC;

In [0]:
SELECT * FROM average_connection_quality;

In [0]:
SELECT * FROM chat_issues order by classification, sentiment;

In [0]:
SELECT * FROM sentiment_for_agent ORDER BY agent_name, sentiment LIMIT 20;